# Computer Vision exercises
Fill in the gaps (`...`). Sections follow the teaching order.

## 1. Convolutional layers
Fill every `...` so the asserts pass. Focus: shapes and the `nn.Conv2d` signature.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# 1a. A conv layer mapping 3 -> 16 channels, 3x3 kernel, halving the spatial size (64 -> 31)
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=0)
x = torch.randn(8, 3, 64, 64)
assert conv(x).shape == (8, 16, 31, 31)

In [8]:
# 1b. Parameter count (with bias) of the layer above. Compute it by hand, then check.
n_params = 3*3*3*16 + 16
print(n_params)
assert n_params == sum(p.numel() for p in conv.parameters())

448


## 2. Image classification
Focus: CNN head, logits vs probabilities, the 5-line training step, eval mode.

In [10]:
import torchvision
from torch.utils.data import DataLoader
from torchvision.transforms import v2

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, n_classes))  # NO softmax here
    def forward(self, x):
        return self.head(self.features(x))

assert SmallCNN()(torch.randn(4, 1, 28, 28)).shape == (4, 10)

In [11]:
# 2b. One training step. Fill the 5 calls in the right order.
def train_step(model, optimizer, criterion, x, y):
    optimizer.zero_grad()  # 1 clear gradients
    logits = model(x)  # 2 forward
    loss = criterion(logits, y)   # 3 loss
    loss.backward()  # 4 backward
    optimizer.step()  # 5 update
    return loss.item()

m = SmallCNN(); opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
l = train_step(m, opt, nn.CrossEntropyLoss(), torch.randn(8, 1, 28, 28), torch.randint(0, 10, (8,)))
assert isinstance(l, float)

In [12]:
# 2c. Accuracy over a dataloader. Note the two lines that turn off training behaviour and gradient tracking.
def accuracy(model, loader, device='cpu'):
    model.eval()          # eval mode
    correct = total = 0
    with torch.no_grad():    # no gradients
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=0)
            correct += (pred == y).sum().item(); total += y.numel()
    return correct / total